# Modélisation ML — CHU Ibn Sina Rabat
## Prophet + Random Forest + XGBoost + SHAP
**PFE BI & Data Science — Azddine 2024/2025**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
import warnings
warnings.filterwarnings('ignore')
import os, joblib

ROOT   = r'C:\Users\Azddine\Desktop\PFE'
GOLD   = os.path.join(ROOT, 'data', 'gold')
MODELS = os.path.join(ROOT, 'models')
os.makedirs(MODELS, exist_ok=True)

print('ROOT :', ROOT)
print('Librairies OK')

## 1. Chargement des données Gold

In [ ]:
urg  = pd.read_csv(os.path.join(GOLD, 'urgences_GOLD.csv'),          encoding='utf-8-sig', low_memory=False)
ts   = pd.read_csv(os.path.join(GOLD, 'serie_temporelle_daily.csv'), encoding='utf-8-sig')

urg['Date_Arrivee'] = pd.to_datetime(urg['Date_Arrivee'])
ts['ds']            = pd.to_datetime(ts['ds'])

print(f'Urgences Gold  : {len(urg):,} lignes')
print(f'Série temporelle : {len(ts):,} jours')
print(f'Période : {ts["ds"].min().date()} → {ts["ds"].max().date()}')

## 2. Feature Engineering — Flux Horaire

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ── Agrégation par heure ──────────────────────────────────
urg['date_h'] = urg['Date_Arrivee'].dt.floor('h')
flux = urg.groupby('date_h').agg(
    nb_patients     = ('ID_Urgence', 'count'),
    nb_medecins     = ('Nb_Medecins_Dispo', 'mean'),
    nb_lits         = ('Nb_Lits_Dispo', 'mean'),
    jour_ferie_moy  = ('Jour_Ferie', 'mean'),
).reset_index()

# ── Features temporelles ─────────────────────────────────
flux['heure']        = flux['date_h'].dt.hour
flux['jour_semaine'] = flux['date_h'].dt.dayofweek
flux['mois']         = flux['date_h'].dt.month
flux['annee']        = flux['date_h'].dt.year
flux['jour_ferie']   = flux['jour_ferie_moy'].round().astype(int)
flux['est_weekend']  = (flux['jour_semaine'] >= 5).astype(int)
flux['trimestre']    = flux['date_h'].dt.quarter
flux['jour_annee']   = flux['date_h'].dt.dayofyear
flux['semaine']      = flux['date_h'].dt.isocalendar().week.astype(int)

# Saison encodée
def saison_num(m):
    if m in [12,1,2]: return 0  # Hiver
    if m in [3,4,5]:  return 1  # Printemps
    if m in [6,7,8]:  return 2  # Eté
    return 3                    # Automne

flux['saison_enc'] = flux['mois'].apply(saison_num)

# ── Features cycliques (sin/cos) pour heure et jour ──────
flux['heure_sin']    = np.sin(2 * np.pi * flux['heure']        / 24)
flux['heure_cos']    = np.cos(2 * np.pi * flux['heure']        / 24)
flux['jour_sin']     = np.sin(2 * np.pi * flux['jour_semaine'] / 7)
flux['jour_cos']     = np.cos(2 * np.pi * flux['jour_semaine'] / 7)
flux['mois_sin']     = np.sin(2 * np.pi * flux['mois']         / 12)
flux['mois_cos']     = np.cos(2 * np.pi * flux['mois']         / 12)

# ── Features lag (décalées) ──────────────────────────────
flux = flux.sort_values('date_h').reset_index(drop=True)
flux['lag_1h']  = flux['nb_patients'].shift(1).fillna(0)
flux['lag_24h'] = flux['nb_patients'].shift(24).fillna(0)
flux['lag_168h']= flux['nb_patients'].shift(168).fillna(0)  # 1 semaine

# ── Rolling moyennes ─────────────────────────────────────
flux['roll_3h']  = flux['nb_patients'].shift(1).rolling(3,  min_periods=1).mean().fillna(0)
flux['roll_24h'] = flux['nb_patients'].shift(1).rolling(24, min_periods=1).mean().fillna(0)
flux['roll_7d']  = flux['nb_patients'].shift(1).rolling(168,min_periods=1).mean().fillna(0)

# ── Pics historiques par (heure, jour, mois) ─────────────
mean_heure = flux.groupby('heure')['nb_patients'].transform('mean')
mean_jour  = flux.groupby('jour_semaine')['nb_patients'].transform('mean')
mean_mois  = flux.groupby('mois')['nb_patients'].transform('mean')
flux['mean_hist_heure'] = mean_heure
flux['mean_hist_jour']  = mean_jour
flux['mean_hist_mois']  = mean_mois

print(f'Dataset flux horaire : {len(flux):,} heures')
print(f'Features créées : {flux.shape[1] - 2} (+ target nb_patients)')
flux[['date_h','nb_patients','heure','lag_24h','roll_7d']].head()

## 3. Split Train / Test (80/20 temporel)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

FEATURES = [
    'heure','jour_semaine','mois','annee','jour_ferie','est_weekend',
    'trimestre','saison_enc','nb_medecins','nb_lits',
    'heure_sin','heure_cos','jour_sin','jour_cos','mois_sin','mois_cos',
    'lag_1h','lag_24h','lag_168h',
    'roll_3h','roll_24h','roll_7d',
    'mean_hist_heure','mean_hist_jour','mean_hist_mois'
]
TARGET = 'nb_patients'

# Split temporel : 80% train, 20% test
split_idx = int(len(flux) * 0.80)
train = flux.iloc[:split_idx]
test  = flux.iloc[split_idx:]

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f'Train : {len(X_train):,} heures  |  Test : {len(X_test):,} heures')
print(f'y_train — moy: {y_train.mean():.2f}  std: {y_train.std():.2f}  max: {y_train.max()}')

# Sauvegarder le LabelEncoder (saison)
le = LabelEncoder()
le.fit(['Hiver','Printemps','Eté','Automne'])
joblib.dump(le, os.path.join(MODELS, 'label_encoder.pkl'))
print('label_encoder.pkl sauvegardé')

## 4. Modèle 1 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print('Entraînement Random Forest...')
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_rf = np.clip(y_pred_rf, 0, None)

r2_rf   = r2_score(y_test, y_pred_rf)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print(f'Random Forest — R²: {r2_rf:.4f}  MAE: {mae_rf:.3f}  RMSE: {rmse_rf:.3f}')

# Sauvegarder
joblib.dump(rf, os.path.join(MODELS, 'random_forest_model.pkl'))
print('random_forest_model.pkl sauvegardé')

## 5. Modèle 2 — XGBoost

In [ ]:
from xgboost import XGBRegressor

print('Entraînement XGBoost...')
xgb = XGBRegressor(
    n_estimators=600,
    max_depth=7,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.05,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = xgb.predict(X_test)
y_pred_xgb = np.clip(y_pred_xgb, 0, None)

r2_xgb   = r2_score(y_test, y_pred_xgb)
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f'XGBoost — R²: {r2_xgb:.4f}  MAE: {mae_xgb:.3f}  RMSE: {rmse_xgb:.3f}')

# Sauvegarder
joblib.dump(xgb, os.path.join(MODELS, 'xgboost_model.pkl'))
print('xgboost_model.pkl sauvegardé')

## 6. Modèle 3 — Prophet (Série Temporelle)

In [ ]:
from prophet import Prophet

print('Entraînement Prophet...')

# Prophet sur données journalières
ts_train = ts[ts['ds'] < '2023-01-01'].copy()
ts_test  = ts[ts['ds'] >= '2023-01-01'].copy()

prophet = Prophet(
    changepoint_prior_scale=0.15,
    seasonality_prior_scale=12.0,
    holidays_prior_scale=10.0,
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)

# Ajouter saisonnalité mensuelle
prophet.add_seasonality(name='monthly', period=30.5, fourier_order=6)

prophet.fit(ts_train)

# Prédictions sur test
future_test = prophet.make_future_dataframe(periods=len(ts_test), freq='D')
forecast_test = prophet.predict(future_test)
forecast_test = forecast_test[forecast_test['ds'].isin(ts_test['ds'])]

y_true_p = ts_test['y'].values
y_pred_p = forecast_test['yhat'].values[:len(y_true_p)]
y_pred_p = np.clip(y_pred_p, 0, None)

r2_p   = r2_score(y_true_p, y_pred_p)
mae_p  = mean_absolute_error(y_true_p, y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y_true_p, y_pred_p))

print(f'Prophet — R²: {r2_p:.4f}  MAE: {mae_p:.3f}  RMSE: {rmse_p:.3f}')

## 7. Prévisions 30 jours (Prophet)

In [ ]:
# Réentraîner Prophet sur toutes les données
prophet_full = Prophet(
    changepoint_prior_scale=0.15,
    seasonality_prior_scale=12.0,
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)
prophet_full.add_seasonality(name='monthly', period=30.5, fourier_order=6)
prophet_full.fit(ts)

future_30 = prophet_full.make_future_dataframe(periods=30, freq='D')
forecast_30 = prophet_full.predict(future_30)

pred_30 = forecast_30[['ds','yhat','yhat_lower','yhat_upper']].tail(30).copy()
pred_30['yhat']       = pred_30['yhat'].clip(0).round(1)
pred_30['yhat_lower'] = pred_30['yhat_lower'].clip(0).round(1)
pred_30['yhat_upper'] = pred_30['yhat_upper'].clip(0).round(1)
pred_30['ds'] = pred_30['ds'].dt.strftime('%Y-%m-%d')

pred_30.to_csv(os.path.join(GOLD, 'predictions_30jours.csv'), index=False, encoding='utf-8-sig')
print('predictions_30jours.csv sauvegardé')
pred_30.head(10)

## 8. SHAP — Importance des features (XGBoost)

In [ ]:
import shap

explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test.sample(500, random_state=42))

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test.sample(500, random_state=42),
    plot_type='bar',
    show=False
)
plt.title('XGBoost — Importance SHAP des features', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'rapports', 'eda', 'shap_importance.png'), dpi=150)
plt.show()
print('SHAP chart sauvegardé')

## 9. Comparaison des modèles

In [ ]:
metrics = pd.DataFrame([
    {'modele': 'Prophet',       'R2': round(r2_p,   4), 'MAE': round(mae_p,   3), 'RMSE': round(rmse_p,   3)},
    {'modele': 'Random Forest', 'R2': round(r2_rf,  4), 'MAE': round(mae_rf,  3), 'RMSE': round(rmse_rf,  3)},
    {'modele': 'XGBoost',       'R2': round(r2_xgb, 4), 'MAE': round(mae_xgb, 3), 'RMSE': round(rmse_xgb, 3)},
])

metrics.to_csv(os.path.join(MODELS, 'metrics_comparison.csv'), index=False, encoding='utf-8-sig')
print('metrics_comparison.csv sauvegardé')
print()
print(metrics.to_string(index=False))

## 10. Visualisation — Prévisions vs Réel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# XGBoost — prédit vs réel
sample = min(500, len(y_test))
axes[0].scatter(y_test.values[:sample], y_pred_xgb[:sample], alpha=0.4, s=10, color='steelblue')
lims = [0, max(y_test.values[:sample].max(), y_pred_xgb[:sample].max()) * 1.05]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_xlabel('Réel')
axes[0].set_ylabel('Prédit')
axes[0].set_title(f'XGBoost — R² = {r2_xgb:.4f}')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)

# Prophet — série temporelle
axes[1].plot(ts['ds'].values[-365:], ts['y'].values[-365:], label='Réel', color='black', linewidth=1)
fc = forecast_30
axes[1].plot(pd.to_datetime(pred_30['ds']), pred_30['yhat'], label='Prévision', color='steelblue', linewidth=2)
axes[1].fill_between(
    pd.to_datetime(pred_30['ds']),
    pred_30['yhat_lower'], pred_30['yhat_upper'],
    alpha=0.3, color='steelblue', label='IC 95%'
)
axes[1].set_title('Prophet — Prévisions 30 jours')
axes[1].legend()

plt.suptitle('Résultats Modélisation ML — CHU Ibn Sina', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'rapports', 'eda', 'resultats_ml.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Vérification finale

In [ ]:
print('=' * 55)
print('BILAN FINAL')
print('=' * 55)

fichiers = {
    'models/random_forest_model.pkl'        : os.path.join(MODELS, 'random_forest_model.pkl'),
    'models/xgboost_model.pkl'              : os.path.join(MODELS, 'xgboost_model.pkl'),
    'models/label_encoder.pkl'              : os.path.join(MODELS, 'label_encoder.pkl'),
    'models/metrics_comparison.csv'         : os.path.join(MODELS, 'metrics_comparison.csv'),
    'data/gold/predictions_30jours.csv'     : os.path.join(GOLD,   'predictions_30jours.csv'),
}

for nom, path in fichiers.items():
    status = 'OK' if os.path.exists(path) else 'MANQUANT'
    print(f'  [{status}] {nom}')

print()
print('METRIQUES :')
print(metrics.to_string(index=False))

best_r2 = metrics['R2'].max()
print()
print(f'Meilleur R² : {best_r2:.4f}', '(objectif >= 0.90)' if best_r2 >= 0.90 else '(en dessous de 0.90)')